# 02 gsMap Analysis: AD GWAS Projection onto Whole-Body Spatial Transcriptomics

## Purpose
Run gsMap `quick_mode` to project Alzheimer's Disease GWAS signals onto per-organ mouse spatial transcriptomics data.

**Paper position**: Core analysis generating results for **Figures 2-5** and **Supplementary Figures S2-S5**.

### Pipeline:
1. Run gsMap quick_mode for each organ (2 control replicates x 16 organs = 32 runs)
2. Collect per-spot p-values and per-annotation Cauchy combination results
3. Aggregate results across organs for whole-body visualization

In [ ]:
import os, glob, subprocess, shlex, time
import pandas as pd
import numpy as np

BASE = '../analysis/26_gsmap'
DATA = f'{BASE}/data'
ST_DIR = f'{DATA}/st/per_organ'
GWAS_FILE = f'{DATA}/gwas/AD_Bellenguez2022.sumstats.gz'
RESOURCE_DIR = f'{DATA}/gsMap_resource'
HOMOLOG_FILE = f'{RESOURCE_DIR}/homologs/mouse_human_homologs.txt'
WORKDIR = f'{BASE}/models/gsmap_output'
GSMAP = '../env/gsmap/bin/gsmap'
os.makedirs(WORKDIR, exist_ok=True)

# List all per-organ h5ad files
h5ad_files = sorted(glob.glob(f'{ST_DIR}/*.h5ad'))
print(f"Found {len(h5ad_files)} organ files:")
for f in h5ad_files:
    print(f"  {os.path.basename(f)}")

In [ ]:
# Run gsMap quick_mode for each organ file
results = {}
for h5ad_path in h5ad_files:
    sample_name = os.path.basename(h5ad_path).replace('.h5ad', '')
    
    # Check if already completed
    ldsc_output = f'{WORKDIR}/{sample_name}/spatial_ldsc/{sample_name}_AD.csv.gz'
    if os.path.exists(ldsc_output):
        print(f"[SKIP] {sample_name}: already completed")
        results[sample_name] = 'completed'
        continue
    
    cmd = (
        f'{GSMAP} quick_mode '
        f'--workdir {WORKDIR} '
        f'--sample_name {sample_name} '
        f'--gsMap_resource_dir {RESOURCE_DIR} '
        f'--hdf5_path {h5ad_path} '
        f'--annotation annotation '
        f'--data_layer count '
        f'--sumstats_file {GWAS_FILE} '
        f'--trait_name AD '
        f'--homolog_file {HOMOLOG_FILE}'
    )
    
    print(f"\n{'='*60}")
    print(f"[RUN] {sample_name}")
    print(f"{'='*60}")
    t0 = time.time()
    
    result = subprocess.run(
        shlex.split(cmd), 
        capture_output=True, text=True, 
        timeout=7200
    )
    
    elapsed = time.time() - t0
    
    if result.returncode == 0:
        print(f"[OK] {sample_name} completed in {elapsed/60:.1f} min")
        results[sample_name] = 'completed'
    else:
        print(f"[FAIL] {sample_name} after {elapsed/60:.1f} min")
        print(f"  STDERR (last 500 chars): {result.stderr[-500:]}")
        results[sample_name] = 'failed'

print(f"\n\nSummary: {sum(v=='completed' for v in results.values())}/{len(results)} completed")

## Collect and aggregate results across all organs

In [ ]:
# Collect spatial LDSC results from all organs
import anndata as ad

all_spot_results = []
all_cauchy_results = []

for h5ad_path in h5ad_files:
    sample_name = os.path.basename(h5ad_path).replace('.h5ad', '')
    
    # Spatial LDSC per-spot results
    ldsc_file = f'{WORKDIR}/{sample_name}/spatial_ldsc/{sample_name}_AD.csv.gz'
    if os.path.exists(ldsc_file):
        df = pd.read_csv(ldsc_file)
        # Load organ info from h5ad obs
        adata_tmp = ad.read_h5ad(h5ad_path, backed='r')
        organ = adata_tmp.obs['Organ_Full_Name'].iloc[0]
        sample = adata_tmp.obs['Sample'].iloc[0]
        df['organ'] = organ
        df['sample'] = sample
        df['sample_name'] = sample_name
        # Add spatial coordinates
        df['x'] = adata_tmp.obsm['spatial'][:, 0]
        df['y'] = adata_tmp.obsm['spatial'][:, 1]
        all_spot_results.append(df)
    
    # Cauchy combination results
    cauchy_file = f'{WORKDIR}/{sample_name}/cauchy_combination/{sample_name}_AD.Cauchy.csv.gz'
    if os.path.exists(cauchy_file):
        df_c = pd.read_csv(cauchy_file)
        df_c['organ'] = organ
        df_c['sample'] = sample
        df_c['sample_name'] = sample_name
        all_cauchy_results.append(df_c)

if all_spot_results:
    spot_df = pd.concat(all_spot_results, ignore_index=True)
    spot_df.to_csv(f'{BASE}/results/all_organs_spot_AD_pvalues.csv.gz', index=False, compression='gzip')
    print(f"Spot-level results: {spot_df.shape[0]} spots from {spot_df['organ'].nunique()} organs")
    print(f"\nPer-organ spot counts:")
    print(spot_df.groupby('organ').size().sort_values(ascending=False))
else:
    print("No spot-level results found yet!")

if all_cauchy_results:
    cauchy_df = pd.concat(all_cauchy_results, ignore_index=True)
    cauchy_df.to_csv(f'{BASE}/results/all_organs_cauchy_AD.csv.gz', index=False, compression='gzip')
    print(f"\nCauchy combination results: {cauchy_df.shape[0]} annotations")
    print(cauchy_df.sort_values('p_cauchy').head(20))
else:
    print("No Cauchy combination results found yet!")